In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

## Install libraries

```bash
conda create -n edu4 python=3.10 jupyter matplotlib
```

```bash 
! pip install -U -r requirements.txt
```

```bash
! pip install -U numpy
! pip install -U scikit-learn
```

In [ ]:
! ls

In [ ]:
! pip install -U -r requirements.txt

## Update repository

In [ ]:
! git pull

## Add import path

In [ ]:
import os
import sys
import gc

In [ ]:
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
del module_path

## Organize imports

In [ ]:
import multiprocessing
from pathlib import Path

In [ ]:
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import iqr
import sklearn
from sklearn import datasets
from sklearn.model_selection import (train_test_split, 
                                     RepeatedStratifiedKFold, cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis, 
                                           QuadraticDiscriminantAnalysis)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, 
                             classification_report, confusion_matrix)

#### Number of CPU cores

In [ ]:
workers = multiprocessing.cpu_count()
workers

## Initialize path

In [ ]:
DATA = Path('data')
PATH = DATA / 'log_regr_lda_qda_np'
PUMPKIN_DIR = PATH / 'Pumpkin_Seeds_Dataset'
IRIS_DIR = PATH / 'iris'
PUMPKIN_DIR.mkdir(exist_ok=True, parents=True)
IRIS_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
! ls

In [ ]:
! ls {PUMPKIN_DIR}

## Download data

In [ ]:
studen_scores_url = 'https://www.kaggle.com/datasets/muratkokludataset/pumpkin-seeds-dataset/download?datasetVersionNumber=1'

In [ ]:
! wget -P {PATH} {studen_scores_url}

## Prepare data

In [ ]:
SEED = 2023

In [ ]:
studen_scores_path = PUMPKIN_DIR / 'Pumpkin_Seeds_Dataset.xlsx'

In [ ]:
df = pd.read_excel(studen_scores_path)
df

In [ ]:
df.shape

## Data analysis

In [ ]:
df['Class'].value_counts() 

In [ ]:
df.describe().T

In [ ]:
sns.pairplot(data=df, hue='Class')

In [ ]:
sns.boxplot(data=df, orient='h') 

In [ ]:
plt.figure(figsize=(15, 10))

correlations = df.corr()
sns.heatmap(correlations, annot=True)

## Pre-processing the Data

In [ ]:
y = df['Class']
X = df.drop(columns=['Class'], axis=1)

In [ ]:
y = y.replace('Çerçevelik', 0).replace('Ürgüp Sivrisi', 1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=.25, 
                                                    random_state=SEED)

In [ ]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

#### Scaling Data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

```python
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
```

In [ ]:
column_names = df.columns[:12] # selecting all columns apart from Class from the original data
X_train = pd.DataFrame(X_train, columns=column_names)

sns.boxplot(data=X_train, orient='h')

#### Removing Outliers with IQR Method

Calculate quantiles and IQR to remove outliers
$$
Minimum = Q1 - 1.5*IQR
$$
and
$$
Maximum = Q3 + 1.5*IQR
$$

<img src="images/quantiles.png">Quartiles</img>

In [ ]:
Q1 = X_train.quantile(q=.25)
Q3 = X_train.quantile(q=.75)

IQR = X_train.apply(iqr)

In [ ]:
# Calculationg minimum and maximum values
minimum = X_train < (Q1-1.5*IQR)
maximum = X_train > (Q3+1.5*IQR)

# The tilde (~) is a reverse operator, 
# and it will select any row that is not below the minimum or above the maximum area, 
# this is our IQR filter
filter = ~(minimum | maximum).any(axis=1)

# We can now select the IQR rows in X_train
X_train = X_train[filter]

In [ ]:
X_train.shape, y_train.shape, y_train.shape[0] - X_train.shape[0]

In [ ]:
y_train = y_train.iloc[X_train.index]

In [ ]:
y_train.shape

## Fit the Logistic Regression Model

In [ ]:
logreg = LogisticRegression(random_state=SEED, 
                            n_jobs=workers)

In [ ]:
# When fitting a DataFrame, rather than a bare NumPy array
# to avoid exceptions, we'll feed only the values without column names
logreg.fit(X_train.values, y_train)

In [ ]:
y_pred = logreg.predict(X_test)

In [ ]:
X_train[:3]

In [ ]:
y_pred[:3] 

In [ ]:
y_pred_proba = logreg.predict_proba(X_test)

In [ ]:
y_pred_proba[:3]

## Evaluating the Model with Classification Reports

Precision:
$$
precision = \frac{\text{true positive}}{\text{true positive} + \text{false positive}}
$$

Recall:
$$
recall = \frac{\text{true positive}}{\text{true positive} + \text{false negative}}
$$

Accuracy:
$$
accuracy = \frac{\text{number of correct predictions}}{\text{total number of predictions}}
$$

f1-score:
$$
\text{f1-score} = 2* \frac{\text{precision} * \text{recall}}{\text{precision} + \text{recall}}
$$



$a = \frac{\sum_{f(x) = y}f(x)}{\sum_{x \in X}{f(x)}}$

In [ ]:
cr = classification_report(y_test, y_pred)
print(cr)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')

In [ ]:
(292 + 251) / (292 + 251 + 49 + 33)

## Estimate coefficients

In [ ]:
logreg.coef_

In [ ]:
logreg.intercept_

Inferencee with Logistic Regression models:

$$
p{X} = \frac{e^{(b_0 + b_1 * x_1 + b_2 * x_2 + b_3 * x_3 + \ldots + b_n * x_n)}}{1 + e^{(b_0 + b_1 * x_1 + b_2 * x_2 + b_3 * x_3 + \ldots + b_n * x_n)}}
$$

## Inference / evaluate the model

In [ ]:
import math

lin_reg = logreg.intercept_[0] + \
((logreg.coef_[0][0]* X_test[:1][0][0])+ \
(logreg.coef_[0][1]* X_test[:1][0][1])+ \
(logreg.coef_[0][2]* X_test[:1][0][2])+ \
(logreg.coef_[0][3]* X_test[:1][0][3])+ \
(logreg.coef_[0][4]* X_test[:1][0][4])+ \
(logreg.coef_[0][5]* X_test[:1][0][5])+ \
(logreg.coef_[0][6]* X_test[:1][0][6])+ \
(logreg.coef_[0][7]* X_test[:1][0][7])+ \
(logreg.coef_[0][8]* X_test[:1][0][8])+ \
(logreg.coef_[0][9]* X_test[:1][0][9])+ \
(logreg.coef_[0][10]* X_test[:1][0][10])+ \
(logreg.coef_[0][11]* X_test[:1][0][11]))

px = math.exp(lin_reg)/(1 +(math.exp(lin_reg)))
1 - px, px

In [ ]:
X_test.shape, logreg.coef_.shape

In [ ]:
logit_v = X_test @ logreg.coef_.T + logreg.intercept_
logit_v

In [ ]:
e_v = np.exp(logit_v)
e_v

In [ ]:
y_pred_v = e_v / (1 + e_v)
y_pred_v

In [ ]:
(1 - y_pred_v[0], y_pred_v[0])

In [ ]:
logreg.predict_proba(X_test[:1])

Recall logits:

$$
ln \left( \frac{p}{1-p} \right)
$$

In [ ]:
lp = logreg.predict_log_proba(X_test[:1])
lp

In [ ]:
np.exp(lp)

## Quadratic and linear discriminant analysis

In [ ]:
SEED = 2023

In [ ]:
iris_url = 'https://www.kaggle.com/datasets/uciml/iris/download?datasetVersionNumber=2'

#### Load dataset

```python
#load iris dataset
iris = datasets.load_iris()

#convert dataset to pandas DataFrame
df = pd.DataFrame(data = np.c_[iris['data'], iris['target']],
                 columns = iris['feature_names'] + ['target'])
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df.columns = ['s_length', 's_width', 'p_length', 'p_width', 'target', 'species']

#view first six rows of DataFrame
df.head(), len(df.index)
```


In [ ]:
! ls {IRIS_DIR}

In [ ]:
df = pd.read_csv(IRIS_DIR / 'Iris.csv')

In [ ]:
df

In [ ]:
df['Species'].value_counts()

In [ ]:
y = df['Species']
X = df.drop(columns=['Id', 'Species'], axis=1)
X.shape, y.shape, df['Species'].value_counts()

```python
# #define predictor and response variables
X = df[['s_length', 's_width', 'p_length', 'p_width']]
y = df['species']
X.shape, y.shape
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

## Train the model

In [ ]:
lda = LinearDiscriminantAnalysis()
qda = QuadraticDiscriminantAnalysis()

In [ ]:
lda = lda.fit(X_train, y_train)
lda

In [ ]:
qda = qda.fit(X_train, y_train)
qda

#### Evluate the model

In [ ]:
y_pred_ln = lda.predict(X_test)

In [ ]:
y_pred_qv = qda.predict(X_test)

In [ ]:
cr = classification_report(y_test, y_pred_ln)
print(cr)

In [ ]:
cr = classification_report(y_test, y_pred_qv)
print(cr)

In [ ]:
cm = confusion_matrix(y_test, y_pred_ln)
sns.heatmap(cm, annot=True, fmt='d')

In [ ]:
cm = confusion_matrix(y_test, y_pred_qv)
sns.heatmap(cm, annot=True, fmt='d')

#### Evaluate the model K-folds CV times

In [ ]:
#Define method to evaluate model
cv_ln = RepeatedStratifiedKFold(
    n_splits=40, n_repeats=12, random_state=SEED)

#evaluate model
scores_nl = cross_val_score(lda, X, y, scoring='accuracy', cv=cv_ln, n_jobs=workers)
print(np.mean(scores_nl)) 

In [ ]:
#Define method to evaluate model
cv_qv = RepeatedStratifiedKFold(
    n_splits=40, n_repeats=12, random_state=SEED)

#evaluate model
scores_qv = cross_val_score(qda, X, y, scoring='accuracy', cv=cv_qv, n_jobs=workers)
print(np.mean(scores_qv)) 